In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("AI_Resume_Screening.csv")

In [3]:
print(df.head())

   Resume_ID              Name                                        Skills  \
0          1        Ashley Ali                      TensorFlow, NLP, Pytorch   
1          2      Wesley Roman  Deep Learning, Machine Learning, Python, SQL   
2          3     Corey Sanchez         Ethical Hacking, Cybersecurity, Linux   
3          4  Elizabeth Carney                   Python, Pytorch, TensorFlow   
4          5        Julie Hill                              SQL, React, Java   

   Experience (Years) Education                Certifications  \
0                  10      B.Sc                           NaN   
1                  10       MBA                     Google ML   
2                   1       MBA  Deep Learning Specialization   
3                   7    B.Tech                 AWS Certified   
4                   4       PhD                           NaN   

                Job Role Recruiter Decision  Salary Expectation ($)  \
0          AI Researcher               Hire              

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Resume_ID               1000 non-null   int64
 1   Name                    1000 non-null   str  
 2   Skills                  1000 non-null   str  
 3   Experience (Years)      1000 non-null   int64
 4   Education               1000 non-null   str  
 5   Certifications          726 non-null    str  
 6   Job Role                1000 non-null   str  
 7   Recruiter Decision      1000 non-null   str  
 8   Salary Expectation ($)  1000 non-null   int64
 9   Projects Count          1000 non-null   int64
 10  AI Score (0-100)        1000 non-null   int64
dtypes: int64(5), str(6)
memory usage: 162.4 KB


In [5]:
df1=pd.read_csv("UpdatedResumeDataSet.csv")

In [6]:
df1.head()

,Category,Resume
0,Data Science,Skills * Programming Languages: Python (pandas...
1,Data Science,Education Details \nMay 2013 to May 2017 B.E ...
2,Data Science,"Areas of Interest Deep Learning, Control Syste..."
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...
4,Data Science,"Education Details \n MCA YMCAUST, Faridabad..."


In [7]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 962 entries, 0 to 961
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Category  962 non-null    str  
 1   Resume    962 non-null    str  
dtypes: str(2)
memory usage: 2.9 MB


In [8]:
df.isnull().sum()

Resume_ID                   0
Name                        0
Skills                      0
Experience (Years)          0
Education                   0
Certifications            274
Job Role                    0
Recruiter Decision          0
Salary Expectation ($)      0
Projects Count              0
AI Score (0-100)            0
dtype: int64

In [9]:
df1.isnull().sum()

Category    0
Resume      0
dtype: int64

In [10]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\d{10,}", " ", text)
    text = re.sub(r"[^a-zA-Z+#.\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [11]:
df1["clean_resume"] = df1["Resume"].apply(clean_text)

df1 = df1.drop_duplicates(subset="clean_resume").reset_index(drop=True)

print(df1.shape)

(166, 3)


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer # changes word into number
from sklearn.linear_model import LogisticRegression # learns category pattern
from sklearn.pipeline import Pipeline # puts steps togetter
from sklearn.metrics import classification_report # prints reportcard for your model

In [13]:
x=df1["Resume"].apply(clean_text)
y=df1["Category"]

In [14]:
x_train,x_test,y_train,y_test=train_test_split(
    x,y, test_size=0.2,random_state=42,stratify=y
)

In [15]:
model= Pipeline([
    ("tfidf",TfidfVectorizer(stop_words="english",max_features=5000)), #pipelines creates the learning path tfidf converts the words into number and the linear regression learns the pattern
    ("classifier",LogisticRegression(max_iter=2000))
])

In [16]:
model.fit(x_train,y_train)
predictions=model.predict(x_test)
print(classification_report(y_test,predictions))

                           precision    recall  f1-score   support

                 Advocate       0.67      1.00      0.80         2
                     Arts       0.00      0.00      0.00         1
       Automation Testing       0.33      1.00      0.50         1
               Blockchain       0.00      0.00      0.00         1
         Business Analyst       0.00      0.00      0.00         1
           Civil Engineer       1.00      1.00      1.00         1
             Data Science       1.00      1.00      1.00         2
                 Database       0.67      1.00      0.80         2
          DevOps Engineer       0.00      0.00      0.00         1
         DotNet Developer       1.00      0.50      0.67         2
            ETL Developer       1.00      1.00      1.00         1
   Electrical Engineering       1.00      1.00      1.00         1
                       HR       0.67      1.00      0.80         2
                   Hadoop       1.00      1.00      1.00     

c:\Users\ADITYA PRTAP SINGH\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADITYA PRTAP SINGH\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADITYA PRTAP SINGH\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to

In [17]:
import joblib
joblib.dump(model, "category_model.pkl")

['category_model.pkl']

In [18]:
SKILLS = [ # skill extractor
    "python", "java", "sql", "machine learning", "deep learning",
    "tensorflow", "pytorch", "pandas", "numpy", "excel", "tableau",
    "power bi", "react", "node.js", "docker", "aws", "html", "css"
]

def extract_skills(text):
    text = text.lower()
    return [skill for skill in SKILLS if skill in text]

In [19]:
extract_skills("I know Python, SQL, Docker and AWS")

['python', 'sql', 'docker', 'aws']

In [20]:

def match_resume_to_job(resume_text, job_description):
    resume_skills = set(extract_skills(resume_text))
    job_skills = set(extract_skills(job_description))

    matched = resume_skills.intersection(job_skills)
    missing = job_skills - resume_skills

    score = 0 if not job_skills else (len(matched) / len(job_skills)) * 100

    return {
        "score": round(score, 2),
        "matched_skills": list(matched),
        "missing_skills": list(missing)
    }

In [21]:
job = "Need Python, SQL, Machine Learning and Tableau"
resume = "I have Python, SQL and Machine Learning experience and tableau"

print(match_resume_to_job(resume, job))

{'score': 100.0, 'matched_skills': ['machine learning', 'tableau', 'python', 'sql'], 'missing_skills': []}


In [22]:
df.sample(5)

,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
3,4,Elizabeth Carney,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,Hire,46848,0,95
841,842,Matthew Mckinney,"Python, Pytorch",5,MBA,AWS Certified,AI Researcher,Hire,101270,7,100
579,580,John Wilson,"Deep Learning, SQL",1,B.Tech,AWS Certified,Data Scientist,Reject,50848,4,50
398,399,Danielle Davis,"Machine Learning, Python, Deep Learning",1,MBA,Deep Learning Specialization,Data Scientist,Reject,79404,4,55
680,681,Jessica Bradford,"Networking, Ethical Hacking",2,PhD,AWS Certified,Cybersecurity Analyst,Hire,106309,7,75


In [23]:
df1.sample(5)

,Category,Resume,clean_resume
5,Data Science,"SKILLS C Basics, IOT, Python, MATLAB, Data Sci...",skills c basics iot python matlab data science...
144,ETL Developer,Education Details \nJanuary 2015 Bachelor of E...,education details january bachelor of engineer...
56,Civil Engineer,Education Details \n B.E in Civil Engineering ...,education details b.e in civil engineering civ...
6,Data Science,Skills â¢ Python â¢ Tableau â¢ Data Visuali...,skills python tableau data visualization r stu...
77,Business Analyst,Key Skills - Requirement Gathering - Requireme...,key skills requirement gathering requirement a...


In [24]:
df["Recruiter Decision"].value_counts()

Recruiter Decision
Hire      812
Reject    188
Name: count, dtype: int64

In [25]:
df["Certifications"] = df["Certifications"].fillna("None")

In [26]:
features = [
    "Skills",
    "Experience (Years)",
    "Education",
    "Certifications",
    "Job Role",
    "Projects Count"
]

target = "Recruiter Decision"

In [27]:
x=df[features]
y=df[target]

x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y
)

In [28]:
from sklearn.compose import ColumnTransformer #combines different cleaning method
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder # convert category words into number
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer # fills empty values
from sklearn.linear_model import LogisticRegression # ml algo model that lears to predict

In [29]:
text_column = "Skills"

category_columns = [
    "Education",
    "Certifications",
    "Job Role"
]

number_columns = [
    "Experience (Years)",
    "Projects Count"
]

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "skills_text",
            TfidfVectorizer(stop_words="english"),
            text_column
        ),
        (
            "categories",
            Pipeline([
                ("fill_missing", SimpleImputer(strategy="most_frequent")),
                ("one_hot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            category_columns
        ),
        (
            "numbers",
            SimpleImputer(strategy="median"),
            number_columns
        )
    ]
)

In [31]:
screening_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

screening_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['Hire','Reject']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['Skills','Experience (Years)','Education','Certifications','Job Role', 'Projects Count']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('skills_text', ...), ('categories', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'dr

In [32]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = screening_model.predict(x_test)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

        Hire       0.99      0.96      0.98       162
      Reject       0.86      0.97      0.91        38

    accuracy                           0.96       200
   macro avg       0.93      0.97      0.95       200
weighted avg       0.97      0.96      0.97       200

[[156   6]
 [  1  37]]


In [33]:
import joblib

joblib.dump(screening_model, "screening_model.pkl")

['screening_model.pkl']

In [34]:
test_candidate = pd.DataFrame([{
    "Skills": "Python, SQL, Pandas, Machine Learning, TensorFlow",
    "Experience (Years)": 3,
    "Education": "B.Tech",
    "Certifications": "Google ML",
    "Job Role": "Data Scientist",
    "Projects Count": 4
}])

print(screening_model.predict(test_candidate))
print(screening_model.predict_proba(test_candidate))

['Hire']
[[0.94592523 0.05407477]]
